# Query Customer Devices from the ExamplePipeline Ontology



This notebook loads the generated RDF/Turtle instance graph and supporting CSVs, then asks: **What customer devices are associated with this customer?**



Before running the notebook, regenerate the graph from the ExamplePipeline root with `python example-pipeline-etl/scripts/etl/export_ontology_rdf.py`.


In [7]:
from pathlib import Path



import pandas as pd

from rdflib import Graph, Literal, Namespace, RDF, XSD





def find_project_root(start: Path) -> Path:

    for candidate in (start, *start.parents):

        if (candidate / "data").is_dir() and (candidate / "example-pipeline-etl").is_dir():

            return candidate

    raise RuntimeError("Run this notebook from inside the ExamplePipeline workspace.")





PROJECT_ROOT = find_project_root(Path.cwd().resolve())

DATA_DIR = PROJECT_ROOT / "data"

TURTLE_PATH = DATA_DIR / "ontology" / "examplepipeline_instances.ttl"

FEATURES_PATH = DATA_DIR / "features" / "features_unified.csv"

PREDICTIONS_PATH = DATA_DIR / "scored" / "predictions_v1.csv"

EP = Namespace("https://examplepipeline.invalid/ontology/")


In [8]:
graph = Graph()

graph.parse(TURTLE_PATH, format="turtle")



print(f"Loaded {len(graph):,} RDF triples from {TURTLE_PATH.name}.")

print(f"Customers in graph: {len(set(graph.subjects(RDF.type, EP.Customer))):,}")


Loaded 49,004 RDF triples from examplepipeline_instances.ttl.
Customers in graph: 1,000


In [3]:
features = pd.read_csv(FEATURES_PATH)

predictions = pd.read_csv(PREDICTIONS_PATH)



print("Feature columns:", features.columns.tolist())

display(features.head(3))

print("Prediction columns:", predictions.columns.tolist())

display(predictions.head(3))


Feature columns: ['customer', 'modem_mac', 'router_mac', 'modem_rx', 'modem_tx', 'cmts_rx', 'cmts_tx', 'router_snr', 'mtr']


,customer,modem_mac,router_mac,modem_rx,modem_tx,cmts_rx,cmts_tx,router_snr,mtr
0,U467522,LY344474,SSX266252,-2.7020,43.1268,-0.4632,56.3465,-11.9230,25.2967
1,S377665,MV807148,GMG327806,-9.8718,43.0214,-1.0819,52.7135,5.4109,25.3798
2,S486049,AL400379,PTY687651,2.1373,46.4392,-1.4721,52.1244,6.5445,25.5368


Prediction columns: ['customer', 'modem_mac', 'router_mac', 'predicted_bad_service']


,customer,modem_mac,router_mac,predicted_bad_service
0,U467522,LY344474,SSX266252,0
1,S377665,MV807148,GMG327806,0
2,S486049,AL400379,PTY687651,0


## Ask the Ontology



Set `customer_id` to any identifier from the feature table. The default is a customer in the current synthetic sample.


In [4]:
customer_id = "U467522"



customer_row = features.loc[features["customer"].eq(customer_id)]

if customer_row.empty:

    raise ValueError(f"{customer_id} is not present in {FEATURES_PATH.name}.")



display(customer_row[["customer", "modem_mac", "router_mac"]])


,customer,modem_mac,router_mac
0,U467522,LY344474,SSX266252


In [ ]:
# A SPARQL query to retrieve the devices associated with a given customer, 
# including their types and identifiers. 
# The query filters for devices that are either modems or routers 
# and retrieves their identifiers if available.

query = """

PREFIX ep: <https://examplepipeline.invalid/ontology/>



SELECT ?device ?device_type ?identifier

WHERE {

  ?customer a ep:Customer ;

            ep:customerIdentifier ?customer_id ;

            ep:hasCustomerDevice ?device .

  ?device a ?device_type .

  FILTER(?device_type IN (ep:Modem, ep:Router))

  OPTIONAL { ?device ep:modemIdentifier ?identifier }

  OPTIONAL { ?device ep:routerIdentifier ?identifier }

}

ORDER BY ?device_type ?identifier

"""



results = graph.query(

    query,

    initBindings={"customer_id": Literal(customer_id, datatype=XSD.string)},

)

devices = pd.DataFrame(

    [

        {

            "customer": customer_id,

            "device_type": str(row.device_type).rsplit("/", 1)[-1],

            "identifier": str(row.identifier),

            "resource": str(row.device),

        }

        for row in results

    ]

)


In [12]:
results



In [10]:
print(f"What customer devices are associated with {customer_id}?")

if devices.empty:

    print("No modem or router resources are associated with this customer.")

else:

    display(devices[["device_type", "identifier", "resource"]])


What customer devices are associated with U467522?


,device_type,identifier,resource
0,Modem,LY344474,https://examplepipeline.invalid/data/modem/LY3...
1,Router,SSX266252,https://examplepipeline.invalid/data/router/SS...
